# S15 – LLM: инференс, prompt engineering и few-shot

В этом ноутбуке делаем следующий логичный шаг после **BERT** и **mini-RAG**:  
переходим к **большой языковой модели (LLM)** и смотрим, как она решает прикладную задачу **без дообучения** – только за счёт правильно сформулированного запроса.

Главная цель ноутбука – показать несколько уровней практической работы с LLM:

- **zero-shot**: даём инструкцию без примеров;
- **few-shot**: добавляем несколько размеченных примеров прямо в prompt;
- **контроль формата ответа**: просим вернуть метку в строгом виде;
- **управление генерацией**: меняем `temperature`, `top_p`, `max_new_tokens`;
- **стресс-тест**: проверяем поведение на пограничных и неоднозначных текстах.

Важно: мы сознательно берём **небольшую instruct-модель** и **маленький учебный набор текстов**, чтобы ноутбук оставался быстро воспроизводимым.

В качестве демо используем задачу **классификации коротких пользовательских сообщений** по 4 классам:

- `bug` – сообщение об ошибке или сбое;
- `feature` – предложение новой функции или улучшения;
- `question` – вопрос, уточнение, просьба объяснить;
- `praise` – положительный отзыв без явного запроса на изменение.

Это просто учебный пример у которого:

1. в задачи понятные метки;
2. она естественно решается через instruction prompt;
3. на ней хорошо видно отличие между **prompt-only** и более специальной настройкой модели, к которой перейдём в следующем ноутбуке.


## 0. План

К концу ноутбука надо уметь:

1. Загружать небольшую instruct-LLM для локального инференса.
2. Формулировать zero-shot prompt для задачи классификации текста.
3. Сравнивать несколько prompt-шаблонов и видеть, как меняется ответ.
4. Добавлять few-shot примеры прямо в prompt.
5. Просить модель возвращать ответ в более строгом формате.
6. Понимать влияние `temperature`, `top_p` и `max_new_tokens`.
7. Выполнять батчевый прогон по набору текстов и собирать сводную таблицу.
8. Видеть типичные ошибки prompt-based подхода и понимать, когда одного prompt engineering уже недостаточно.


## 1. Импорты и общие настройки

Ниже подключаем библиотеки, фиксируем `seed`, выбираем устройство и загружаем небольшую instruct-модель.

Для снижения требований к памяти предусмотрена **опциональная 4-bit загрузка** через `bitsandbytes`, если пакет доступен и есть GPU.  
Если `bitsandbytes` недоступен, ноутбук всё равно работает в обычном режиме.

> Предполагается, что в окружении уже доступны `torch`, `transformers`, `datasets`, `peft`, `accelerate`, `scikit-learn`, `pandas`, `numpy` и `matplotlib`.  
> Пакет `bitsandbytes` желателен, но не обязателен.


In [1]:
# Базовые библиотеки для воспроизводимости, работы с таблицами и локального инференса LLM.
import os
import random
import re
import warnings
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)
pd.set_option("display.precision", 4)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Подавляем FutureWarning из bitsandbytes, не влияющий на работу ноутбука.
warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")

try:
    import bitsandbytes  # noqa: F401  – проверяем, что пакет реально установлен
    from transformers import BitsAndBytesConfig
    BNB_AVAILABLE = True
except Exception:
    BitsAndBytesConfig = None
    BNB_AVAILABLE = False

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes available:", BNB_AVAILABLE)


Torch: 2.10.0+cu128
CUDA available: True
bitsandbytes available: False


In [2]:
# Фиксируем seed, определяем устройство и загружаем instruct-модель.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", DEVICE)

# Небольшая instruct-модель: достаточно компактна для учебного инференса.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


def load_llm_bundle(model_name: str) -> Tuple[AutoTokenizer, AutoModelForCausalLM, str]:
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    quantization_used = "none"
    model_kwargs: Dict[str, Any] = {"trust_remote_code": True}

    if DEVICE == "cuda":
        model_kwargs["device_map"] = "auto"
        model_kwargs["torch_dtype"] = torch.float16
    else:
        model_kwargs["torch_dtype"] = torch.float32

    if DEVICE == "cuda" and BNB_AVAILABLE and BitsAndBytesConfig is not None:
        try:
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            quantization_used = "4-bit"
        except Exception as e:
            print("Не удалось подготовить 4-bit конфигурацию, продолжаем без неё.")
            print("Причина:", repr(e))
            model_kwargs.pop("quantization_config", None)
            quantization_used = "none"

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()

    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    if getattr(model.generation_config, "pad_token_id", None) is None and tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    return tokenizer, model, quantization_used


tokenizer, model, quantization_used = load_llm_bundle(MODEL_NAME)

print("Model loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model class:", model.__class__.__name__)
print("Quantization:", quantization_used)
print("Pad token id:", tokenizer.pad_token_id)
print("EOS token id:", tokenizer.eos_token_id)


Устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Tokenizer class: Qwen2Tokenizer
Model class: Qwen2ForCausalLM
Quantization: none
Pad token id: 151643
EOS token id: 151645


## 2. Модель, учебный набор текстов и постановка задачи

Используем **небольшой учебный набор коротких сообщений**, похожих на реальные обращения пользователей к сервису или комментарии в issue-трекере.

Для каждого сообщения заранее зададим **эталонную метку**.  
Она нужна не потому, что LLM обязана совпасть с ней на 100%, а чтобы мы могли:

- сравнивать разные prompt-шаблоны;
- оценивать, когда few-shot помогает;
- видеть типичные ошибки и пограничные кейсы.

Отдельно фиксируем 4 допустимых класса:

- `bug`
- `feature`
- `question`
- `praise`


In [3]:
ALLOWED_LABELS = ["bug", "feature", "question", "praise"]

label_descriptions = {
    "bug": "сообщение об ошибке, сбое, падении, некорректной работе",
    "feature": "предложение добавить функцию, улучшение или изменение поведения",
    "question": "вопрос, просьба объяснить, уточнение или запрос информации",
    "praise": "положительный отзыв или благодарность без запроса на изменение",
}

demo_samples: List[Dict[str, str]] = [
    {
        "text": "После обновления приложение вылетает сразу после входа в аккаунт.",
        "target_label": "bug",
    },
    {
        "text": "Было бы удобно добавить тёмную тему и горячие клавиши для навигации.",
        "target_label": "feature",
    },
    {
        "text": "Подскажите, где в личном кабинете можно скачать закрывающие документы?",
        "target_label": "question",
    },
    {
        "text": "Очень понравился новый интерфейс: стало заметно быстрее и чище.",
        "target_label": "praise",
    },
    {
        "text": "Сервис полезный, но иногда страница отчёта грузится слишком долго.",
        "target_label": "bug",
    },
    {
        "text": "Спасибо, теперь экспорт в Excel работает без ошибок и в нужном формате.",
        "target_label": "praise",
    },
    {
        "text": "Почему уведомления приходят только в веб-версии, а в мобильной их нет?",
        "target_label": "question",
    },
    {
        "text": "Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",
        "target_label": "feature",
    },
]

demo_df = pd.DataFrame(demo_samples)
display(demo_df)

print("Допустимые классы:", ALLOWED_LABELS)


,text,target_label
0,После обновления приложение вылетает сразу после входа в аккаунт.,bug
1,Было бы удобно добавить тёмную тему и горячие клавиши для навигации.,feature
2,"Подскажите, где в личном кабинете можно скачать закрывающие документы?",question
3,Очень понравился новый интерфейс: стало заметно быстрее и чище.,praise
4,"Сервис полезный, но иногда страница отчёта грузится слишком долго.",bug
5,"Спасибо, теперь экспорт в Excel работает без ошибок и в нужном формате.",praise
6,"Почему уведомления приходят только в веб-версии, а в мобильной их нет?",question
7,"Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",feature


Допустимые классы: ['bug', 'feature', 'question', 'praise']


In [4]:
SYSTEM_PROMPT = (
    "Ты помогаешь классифицировать короткие пользовательские сообщения по одному из "
    "четырёх классов: bug, feature, question, praise."
)


def build_user_prompt(text: str, style: str = "strict") -> str:
    if style == "minimal":
        return (
            "Определи класс сообщения. "
            "Возможные классы: bug, feature, question, praise.\n\n"
            f"Сообщение: {text}"
        )

    if style == "strict":
        return (
            "Классифицируй сообщение ровно по одному классу из списка: "
            "bug, feature, question, praise.\n"
            "Опирайся на смысл сообщения.\n\n"
            "Правила:\n"
            "- bug: ошибка, сбой, падение, некорректная работа;\n"
            "- feature: просьба добавить функцию или улучшение;\n"
            "- question: вопрос, уточнение, просьба объяснить;\n"
            "- praise: положительный отзыв без запроса на изменение.\n\n"
            f"Сообщение: {text}\n\n"
            "Ответь только названием класса."
        )

    if style == "rubric":
        return (
            "Ниже четыре класса пользовательских сообщений.\n"
            "1) bug – есть явный признак сбоя, падения, ошибки, зависания или некорректной работы.\n"
            "2) feature – есть просьба добавить функцию, фильтр, опцию, настройку или улучшение.\n"
            "3) question – пользователь задаёт вопрос, просит объяснить или уточнить.\n"
            "4) praise – пользователь хвалит продукт или благодарит, не прося ничего изменить.\n\n"
            f"Сообщение: {text}\n\n"
            "Верни только одну метку из набора: bug, feature, question, praise."
        )

    raise ValueError(f"Неизвестный стиль prompt: {style}")


FEW_SHOT_EXAMPLES = [
    {
        "text": "После нажатия на кнопку оплаты форма зависает и ничего не происходит.",
        "label": "bug",
    },
    {
        "text": "Сделайте, пожалуйста, возможность сохранять шаблоны отчётов.",
        "label": "feature",
    },
    {
        "text": "Где посмотреть историю изменений по заявке?",
        "label": "question",
    },
    {
        "text": "Спасибо команде, новый релиз стал заметно стабильнее.",
        "label": "praise",
    },
]


In [5]:
def make_chat_messages(
    user_prompt: str,
    system_prompt: str = SYSTEM_PROMPT,
    few_shot_examples: Optional[List[Dict[str, str]]] = None,
    assistant_prefill: Optional[str] = None,
) -> List[Dict[str, str]]:
    messages: List[Dict[str, str]] = [{"role": "system", "content": system_prompt}]

    if few_shot_examples:
        for example in few_shot_examples:
            messages.append({"role": "user", "content": build_user_prompt(example["text"], style="strict")})
            messages.append({"role": "assistant", "content": example["label"]})

    messages.append({"role": "user", "content": user_prompt})

    if assistant_prefill is not None:
        messages.append({"role": "assistant", "content": assistant_prefill})

    return messages


def generate_from_messages(
    messages: List[Dict[str, str]],
    max_new_tokens: int = 32,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> str:
    do_sample = temperature > 0

    continue_final_message = bool(messages and messages[-1]["role"] == "assistant")

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=not continue_final_message,
        continue_final_message=continue_final_message,
    )

    model_inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        add_special_tokens=False,
    )

    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
    prompt_length = model_inputs["input_ids"].shape[1]

    with torch.no_grad():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[:, prompt_length:]
    text = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
    return text.strip()


def extract_first_label(raw_text: str, allowed_labels: List[str] = ALLOWED_LABELS) -> Optional[str]:
    cleaned = raw_text.strip().lower()

    if cleaned in allowed_labels:
        return cleaned

    match = re.search(r"\b(" + "|".join(re.escape(label) for label in allowed_labels) + r")\b", cleaned)
    if match:
        return match.group(1)

    return None


def predict_label(
    text: str,
    prompt_style: str = "strict",
    few_shot_examples: Optional[List[Dict[str, str]]] = None,
    max_new_tokens: int = 24,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> Dict[str, Any]:
    user_prompt = build_user_prompt(text, style=prompt_style)
    messages = make_chat_messages(
        user_prompt=user_prompt,
        few_shot_examples=few_shot_examples,
    )
    raw_output = generate_from_messages(
        messages,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    parsed_label = extract_first_label(raw_output)

    return {
        "text": text,
        "raw_output": raw_output,
        "parsed_label": parsed_label,
    }


## 3. Первый zero-shot инференс

Начнём с самого простого сценария:  
дадим модели **один текст**, **один instruction prompt** и **никаких примеров**.

Это и есть **zero-shot** режим: модель должна опереться только на свои предобученные знания и на формулировку текущего запроса.

Сразу важно смотреть не только на итоговую метку, но и на:

- сырой ответ модели;
- то, соблюла ли она требуемый формат;
- удалось ли нам корректно извлечь класс из ответа.


In [6]:
zero_shot_examples = demo_df.head(4).copy()

zero_shot_rows = []
for text in zero_shot_examples["text"]:
    result = predict_label(text, prompt_style="strict", few_shot_examples=None)
    zero_shot_rows.append(result)

zero_shot_df = pd.DataFrame(zero_shot_rows)
display(zero_shot_df)


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,text,raw_output,parsed_label
0,После обновления приложение вылетает сразу после входа в аккаунт.,feature,feature
1,Было бы удобно добавить тёмную тему и горячие клавиши для навигации.,feature,feature
2,"Подскажите, где в личном кабинете можно скачать закрывающие документы?",question,question
3,Очень понравился новый интерфейс: стало заметно быстрее и чище.,feature,feature


## 4. Как меняется ответ при изменении prompt-шаблона

Теперь проверим важную инженерную идею:  
**LLM чувствительна к формулировке prompt-а**.

Возьмём три варианта:

- `minimal` – очень короткая инструкция;
- `strict` – явные правила и требование вернуть только метку;
- `rubric` – более развёрнутое описание классов.

Даже если задача формально та же самая, качество и формат ответа могут различаться.


In [7]:
comparison_texts = [
    "После обновления приложение вылетает сразу после входа в аккаунт.",
    "Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",
    "Очень понравился новый интерфейс: стало заметно быстрее и чище.",
]

prompt_rows = []
for text in comparison_texts:
    for prompt_style in ["minimal", "strict", "rubric"]:
        result = predict_label(text, prompt_style=prompt_style)
        prompt_rows.append(
            {
                "text": text,
                "prompt_style": prompt_style,
                "raw_output": result["raw_output"],
                "parsed_label": result["parsed_label"],
            }
        )

prompt_comparison_df = pd.DataFrame(prompt_rows)
display(prompt_comparison_df)


,text,prompt_style,raw_output,parsed_label
0,После обновления приложение вылетает сразу после входа в аккаунт.,minimal,question,question
1,После обновления приложение вылетает сразу после входа в аккаунт.,strict,feature,feature
2,После обновления приложение вылетает сразу после входа в аккаунт.,rubric,bug,bug
3,"Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",minimal,question,question
4,"Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",strict,feature,feature
5,"Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",rubric,feature,feature
6,Очень понравился новый интерфейс: стало заметно быстрее и чище.,minimal,feature,feature
7,Очень понравился новый интерфейс: стало заметно быстрее и чище.,strict,feature,feature
8,Очень понравился новый интерфейс: стало заметно быстрее и чище.,rubric,feature,feature


## 5. Few-shot prompting

Zero-shot полезен как baseline, но часто его недостаточно.  
Следующий шаг – **few-shot prompting**: прямо в prompt добавляем несколько коротких размеченных примеров.

Идея простая:

- мы не меняем веса модели;
- не запускаем обучение;
- но показываем, **какой именно формат и тип решения** от неё ожидаем.

Это особенно полезно, если классы близки по смыслу или формулировка задачи недостаточно очевидна.


In [8]:
few_shot_eval_rows = []

for row in demo_samples:
    zero_result = predict_label(
        row["text"],
        prompt_style="strict",
        few_shot_examples=None,
    )
    few_result = predict_label(
        row["text"],
        prompt_style="strict",
        few_shot_examples=FEW_SHOT_EXAMPLES,
    )

    few_shot_eval_rows.append(
        {
            "text": row["text"],
            "target_label": row["target_label"],
            "zero_shot_label": zero_result["parsed_label"],
            "few_shot_label": few_result["parsed_label"],
            "zero_shot_match": zero_result["parsed_label"] == row["target_label"],
            "few_shot_match": few_result["parsed_label"] == row["target_label"],
        }
    )

few_shot_eval_df = pd.DataFrame(few_shot_eval_rows)
display(few_shot_eval_df)

summary_df = pd.DataFrame(
    {
        "mode": ["zero-shot", "few-shot"],
        "accuracy_like_score": [
            few_shot_eval_df["zero_shot_match"].mean(),
            few_shot_eval_df["few_shot_match"].mean(),
        ],
    }
)
display(summary_df)


,text,target_label,zero_shot_label,few_shot_label,zero_shot_match,few_shot_match
0,После обновления приложение вылетает сразу после входа в аккаунт.,bug,feature,bug,False,True
1,Было бы удобно добавить тёмную тему и горячие клавиши для навигации.,feature,feature,feature,True,True
2,"Подскажите, где в личном кабинете можно скачать закрывающие документы?",question,question,question,True,True
3,Очень понравился новый интерфейс: стало заметно быстрее и чище.,praise,feature,feature,False,False
4,"Сервис полезный, но иногда страница отчёта грузится слишком долго.",bug,feature,feature,False,False
5,"Спасибо, теперь экспорт в Excel работает без ошибок и в нужном формате.",praise,feature,praise,False,True
6,"Почему уведомления приходят только в веб-версии, а в мобильной их нет?",question,feature,feature,False,False
7,"Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",feature,feature,feature,True,True


,mode,accuracy_like_score
0,zero-shot,0.375
1,few-shot,0.625


## 6. Управление форматом ответа

В прикладных системах мало получить «примерно правильный» ответ.  
Нам важно, чтобы ответ был **пригоден для дальнейшей обработки**:

- для таблицы;
- для API;
- для логирования;
- для последующего анализа ошибок.

Попробуем два подхода:

1. обычный свободный ответ;
2. более строгий формат JSON.

Для JSON дополнительно используем **prefill**: заранее начинаем ответ модели с `{"label": "` и смотрим, помогает ли это удерживать формат.


In [9]:
def predict_json_like(
    text: str,
    few_shot_examples: Optional[List[Dict[str, str]]] = None,
) -> Dict[str, Any]:
    user_prompt = (
        "Классифицируй сообщение по одному классу из набора bug, feature, question, praise.\n"
        "Верни JSON со структурой:\n"
        '{"label": "...", "short_reason": "..."}\n\n'
        f"Сообщение: {text}"
    )

    messages = make_chat_messages(
        user_prompt=user_prompt,
        few_shot_examples=few_shot_examples,
        assistant_prefill='{"label": "',
    )

    generated_tail = generate_from_messages(
        messages,
        max_new_tokens=48,
        temperature=0.0,
        top_p=1.0,
    )

    raw_output = '{"label": "' + generated_tail
    parsed_label = extract_first_label(raw_output)

    return {
        "text": text,
        "raw_output": raw_output,
        "parsed_label": parsed_label,
    }


format_rows = []
for text in demo_df.head(4)["text"]:
    free_form = predict_label(text, prompt_style="strict", few_shot_examples=FEW_SHOT_EXAMPLES)
    json_like = predict_json_like(text, few_shot_examples=FEW_SHOT_EXAMPLES)

    format_rows.append(
        {
            "text": text,
            "free_form_output": free_form["raw_output"],
            "free_form_label": free_form["parsed_label"],
            "json_like_output": json_like["raw_output"],
            "json_like_label": json_like["parsed_label"],
        }
    )

format_df = pd.DataFrame(format_rows)
display(format_df)


,text,free_form_output,free_form_label,json_like_output,json_like_label
0,После обновления приложение вылетает сразу после входа в аккаунт.,bug,bug,"{""label"": ""bug"", ""short_reason"": ""приложение вылетает сразу после входа в аккаунт""}",bug
1,Было бы удобно добавить тёмную тему и горячие клавиши для навигации.,feature,feature,"{""label"": ""feature"", ""short_reason"": ""Добавление тёмной темы и горячих клавиши для навигации.""}",feature
2,"Подскажите, где в личном кабинете можно скачать закрывающие документы?",question,question,"{""label"": ""question"", ""short_reason"": ""Необходимо найти информацию о доступности закрывающих документов.""}",question
3,Очень понравился новый интерфейс: стало заметно быстрее и чище.,feature,feature,"{""label"": ""feature"", ""short_reason"": ""интерфейс стал заметно быстрее и чище""}",feature


## 7. Параметры генерации: `temperature`, `top_p`, `max_new_tokens`

Даже на одной и той же задаче ответ LLM зависит не только от prompt-а, но и от параметров генерации.

Ниже сравним несколько режимов:

- **почти детерминированный** (`temperature=0.0`);
- **умеренно стохастический** (`temperature=0.7`, `top_p=0.9`);
- **более свободная генерация** (`temperature=1.0`, `top_p=0.95`).

Для наглядности возьмём более неоднозначный текст, где есть и похвала, и намёк на проблему.


In [10]:
ambiguous_text = "Сервис полезный и идея отличная, но иногда страница отчёта грузится слишком долго."

generation_settings = [
    {"temperature": 0.0, "top_p": 1.0, "max_new_tokens": 16},
    {"temperature": 0.7, "top_p": 0.9, "max_new_tokens": 16},
    {"temperature": 1.0, "top_p": 0.95, "max_new_tokens": 24},
]

generation_rows = []
for cfg in generation_settings:
    result = predict_label(
        ambiguous_text,
        prompt_style="strict",
        few_shot_examples=FEW_SHOT_EXAMPLES,
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        max_new_tokens=cfg["max_new_tokens"],
    )
    generation_rows.append(
        {
            "text": ambiguous_text,
            "temperature": cfg["temperature"],
            "top_p": cfg["top_p"],
            "max_new_tokens": cfg["max_new_tokens"],
            "raw_output": result["raw_output"],
            "parsed_label": result["parsed_label"],
        }
    )

generation_df = pd.DataFrame(generation_rows)
display(generation_df)


,text,temperature,top_p,max_new_tokens,raw_output,parsed_label
0,"Сервис полезный и идея отличная, но иногда страница отчёта грузится слишком долго.",0.0,1.00,16,feature,feature
1,"Сервис полезный и идея отличная, но иногда страница отчёта грузится слишком долго.",0.7,0.90,16,feature,feature
2,"Сервис полезный и идея отличная, но иногда страница отчёта грузится слишком долго.",1.0,0.95,24,feature,feature


## 8. Батчевый прогон и сводная таблица результатов

Теперь соберём **единый табличный результат** по всему учебному набору.

Именно такой формат нужен в реальной работе, когда мы хотим:

- сравнить несколько prompt-стратегий;
- увидеть ошибки модели;
- сохранить результаты в CSV;
- быстро оценить, где few-shot реально дал прирост.


In [11]:
def run_batch_inference(
    samples: List[Dict[str, str]],
    prompt_style: str = "strict",
    few_shot_examples: Optional[List[Dict[str, str]]] = None,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_new_tokens: int = 24,
) -> pd.DataFrame:
    rows = []

    for sample in samples:
        result = predict_label(
            sample["text"],
            prompt_style=prompt_style,
            few_shot_examples=few_shot_examples,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
        )
        rows.append(
            {
                "text": sample["text"],
                "target_label": sample["target_label"],
                "predicted_label": result["parsed_label"],
                "raw_output": result["raw_output"],
                "is_correct": result["parsed_label"] == sample["target_label"],
            }
        )

    return pd.DataFrame(rows)


batch_df = run_batch_inference(
    demo_samples,
    prompt_style="strict",
    few_shot_examples=FEW_SHOT_EXAMPLES,
)

display(batch_df)
print("Доля совпадений с эталонной разметкой:", round(batch_df["is_correct"].mean(), 4))


,text,target_label,predicted_label,raw_output,is_correct
0,После обновления приложение вылетает сразу после входа в аккаунт.,bug,bug,bug,True
1,Было бы удобно добавить тёмную тему и горячие клавиши для навигации.,feature,feature,feature,True
2,"Подскажите, где в личном кабинете можно скачать закрывающие документы?",question,question,question,True
3,Очень понравился новый интерфейс: стало заметно быстрее и чище.,praise,feature,feature,False
4,"Сервис полезный, но иногда страница отчёта грузится слишком долго.",bug,feature,feature,False
5,"Спасибо, теперь экспорт в Excel работает без ошибок и в нужном формате.",praise,praise,praise,True
6,"Почему уведомления приходят только в веб-версии, а в мобильной их нет?",question,feature,feature,False
7,"Добавьте, пожалуйста, фильтр по дате и сохранённые наборы параметров поиска.",feature,feature,feature,True


Доля совпадений с эталонной разметкой: 0.625


## 9. Стресс-тест на пограничных и неоднозначных примерах

Настоящая ценность prompt-based подхода видна не на очевидных кейсах, а на сложных:

- смешанный сигнал: похвала + жалоба;
- скрытый запрос на улучшение;
- формулировка вежливая, но по сути это bug;
- риторический вопрос, который по смыслу ближе к жалобе.

Именно такие примеры быстро показывают пределы zero-shot и few-shot решения.


In [12]:
stress_test_samples = [
    "В целом всё нравится, но после загрузки большого файла интерфейс иногда зависает.",
    "Было бы здорово, если бы можно было назначать права доступа по шаблону.",
    "Не очень понимаю, почему экспорт иногда есть, а иногда кнопка просто неактивна.",
    "Спасибо за релиз, хотя мобильная версия всё ещё открывается медленно.",
    "Можно ли как-то скрыть завершённые задачи из общего списка?",
    "Форма красивая, но при отправке дважды показывает одну и ту же ошибку.",
]

stress_rows = []
for text in stress_test_samples:
    zero_result = predict_label(text, prompt_style="strict", few_shot_examples=None)
    few_result = predict_label(text, prompt_style="strict", few_shot_examples=FEW_SHOT_EXAMPLES)

    stress_rows.append(
        {
            "text": text,
            "zero_shot_output": zero_result["raw_output"],
            "zero_shot_label": zero_result["parsed_label"],
            "few_shot_output": few_result["raw_output"],
            "few_shot_label": few_result["parsed_label"],
        }
    )

stress_df = pd.DataFrame(stress_rows)
display(stress_df)


,text,zero_shot_output,zero_shot_label,few_shot_output,few_shot_label
0,"В целом всё нравится, но после загрузки большого файла интерфейс иногда зависает.",feature,feature,feature,feature
1,"Было бы здорово, если бы можно было назначать права доступа по шаблону.",feature,feature,feature,feature
2,"Не очень понимаю, почему экспорт иногда есть, а иногда кнопка просто неактивна.",feature,feature,question,question
3,"Спасибо за релиз, хотя мобильная версия всё ещё открывается медленно.",question,question,question,question
4,Можно ли как-то скрыть завершённые задачи из общего списка?,feature,feature,feature,feature
5,"Форма красивая, но при отправке дважды показывает одну и ту же ошибку.",feature,feature,bug,bug


## 10. Типичные ошибки prompt-based подхода

На этом этапе уже можно увидеть характерные проблемы prompt engineering без дообучения:

### 10.1. Лишний текст вместо одной метки
Даже если мы просим вернуть только класс, модель может добавить пояснение, вежливую фразу или короткое обоснование.

### 10.2. Нестабильность на пограничных кейсах
На смешанных сообщениях метка может меняться при небольшом изменении prompt-а или параметров генерации.

### 10.3. Смешение ближайших классов
Например, `question` и `feature` нередко пересекаются:  
«Можно ли добавить фильтр?» формально выглядит как вопрос, но по смыслу ближе к запросу на улучшение.

### 10.4. Ошибки формата
JSON или иной структурированный формат может нарушаться, особенно у небольших моделей.

### 10.5. Отсутствие гарантии на доменной лексике
Если сообщения содержат жаргон, внутренние термины продукта или слишком специфичный контекст, zero-shot поведение становится менее предсказуемым.

Ниже отдельно посмотрим на случаи, где модель либо ошиблась по метке, либо вообще не вернула распознаваемый класс.


In [13]:
problem_df = batch_df[
    batch_df["predicted_label"].isna() | (~batch_df["is_correct"])
].copy()

if len(problem_df) == 0:
    print("На текущем учебном наборе все ответы распарсились и совпали с эталонной разметкой.")
else:
    display(problem_df)


,text,target_label,predicted_label,raw_output,is_correct
3,Очень понравился новый интерфейс: стало заметно быстрее и чище.,praise,feature,feature,False
4,"Сервис полезный, но иногда страница отчёта грузится слишком долго.",bug,feature,feature,False
6,"Почему уведомления приходят только в веб-версии, а в мобильной их нет?",question,feature,feature,False


## 11. Ограничения готовой LLM без адаптации

Важно зафиксировать, чего **не стоит ожидать** от prompt-only решения.

### 11.1. Нет гарантии строгого формата
Даже хороший prompt не превращает генеративную модель в идеально детерминированный классификатор.

### 11.2. Качество сильно зависит от wording
Меняется формулировка инструкции – может меняться и ответ.

### 11.3. Few-shot помогает, но не заменяет обучение
Несколько примеров в prompt улучшают поведение модели, но не создают устойчивого специализированного решения.

### 11.4. Стоимость инференса выше, чем у обычного классификатора
Даже компактная LLM тяжелее и медленнее, чем небольшая модель sequence classification.

### 11.5. Доменная адаптация остаётся открытой задачей
Если данные специфичны, классы нетривиальны, а формат ответа критичен, следующим шагом становится **адаптация модели**, а не бесконечное переписывание prompt-а.

Именно этим займёмся в следующем ноутбуке:  
перейдём от prompt-only подхода к **PEFT / LoRA-адаптации**.


## 12. Итоги

1. **LLM можно использовать для прикладной классификации даже без дообучения.**
2. **Zero-shot – это естественный baseline, но он чувствителен к формулировке prompt-а.**
3. **Few-shot обычно делает поведение модели более управляемым, не меняя её веса.**
4. **Строгий формат ответа требует отдельного инженерного внимания.**
5. **Параметры генерации влияют на стабильность результата, особенно на пограничных кейсах.**
6. **Prompt engineering полезен, но у него есть предел – дальше нужна адаптация модели под задачу.**


## Задания для самостоятельной работы

1. Добавьте 10 собственных сообщений и сравните `zero-shot` и `few-shot` режимы на них.
2. Придумайте ещё 5 пограничных кейсов, где сложно различить `question` и `feature`, и проверьте, какой prompt работает лучше.
3. Попробуйте изменить описание классов в `build_user_prompt` и посмотрите, как это влияет на итоговые метки.
4. Реализуйте более строгий JSON-парсер и измерьте, в скольких случаях модель действительно возвращает корректную структуру.
5. Сравните несколько instruct-моделей и оцените, меняется ли качество на одном и том же наборе сообщений.
6. Подготовьте собственный мини-датасет для следующего ноутбука, где мы будем адаптировать LLM через PEFT / LoRA.
